In [ ]:
import numpy as np
import glob
import os
from tqdm.notebook import tqdm
import scipy.linalg as la
from numpy.linalg import matrix_power
import math
import scipy.stats as sp
from statsmodels.stats.multitest import multipletests
import matplotlib as mlt
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import spearmanr

cmap = sns.color_palette("colorblind")

In [ ]:
def graph_metrics(adj_mat): 

    def eigenvector_centrality_und(adj_mat): 
        # Compute eigen vector centrality from undirected graph
        if np.isnan(np.min(adj_mat)): # Skip subjects with nan edges
            centrality = np.nan 
        else: 
            w, v = la.eig(adj_mat)
            idx = np.argmax(w)
            centrality = np.abs(v[idx])
        return centrality

    def clustering_coef_wu(adj_mat): 
        # Compute clustering coefficient from weighted undirected graph
        K = np.sum(adj_mat != 0, axis=1)            	
        cyc3 = np.diag(matrix_power(adj_mat**(1/3),3));           
        if len(np.where(cyc3 == 0)[0]) == 0: 
            CC = cyc3/(K*(K-1))
        else:
            K[np.where(cyc3 == 0)[0]] = np.nan
        return CC

    def strengths_und(adj_mat):
        str_und = np.sum(adj_mat, axis = 1)
        return str_und
    
    centrality = eigenvector_centrality_und(adj_mat)
    CC = clustering_coef_wu(adj_mat)
    str_und = strengths_und(adj_mat)
    
    return centrality, CC, str_und


def create_frequency_axis(f_min=2, f_max= 4, scale_freq ='log'):
    
    if scale_freq == 'log':
        f_min_log = math.floor(np.log10(f_min))
        f_max_log = math.ceil(np.log10(f_max))
        mb = np.logspace(f_min_log, f_max_log, num=40) # Morlet bank 
        freq_vals = mb[(2.1 < mb) & (mb < 75)] # Frequency of interest
        
    
    elif scale_freq == 'lin':
        freq_vals = np.linspace(f_min, f_max, f_max-1)

    freq_labels = [('%.2f'%x) for x in freq_vals]
    
    return freq_vals, freq_labels

def extract_fei_new(files):
    
    data, name_subjs, data_masked = list(), list(), list()
   
    for ifile, file in enumerate(tqdm(files)):
        
        name_subjs.append(file.split('/')[9][:6])
        
        tmp = np.load(file, allow_pickle=True)

        data.append(tmp[1]) 
        data_masked.append(np.nanmean(tmp[1,...],axis=0)) # tmp[1] == rimuovo fEI se 0.5 < cent < 1 non è rispettato
    
    fEI_ch = np.array(data)
    fEI_masked = np.array(data_masked)

    return name_subjs, fEI_ch, fEI_masked

def extract_wpli_new(files):
    
    name_subj, wpli_obs = list(), list()

    for ifile, file in enumerate(tqdm(files)):
        
        data = np.load(file, allow_pickle=True)
        
        _, nCh, _, nF = data.shape

        for d in data:
            for fIdx in range(nF):
                np.fill_diagonal(d[...,fIdx], 0)
                
        _, _, obs, _, _, _= data

        obs_wpli = np.zeros((nCh,nCh, nF))
         
        for iF in range(nF):
            
            obs_wpli[...,iF] = np.abs(obs[...,iF])

        wpli_obs.append(obs_wpli)
    
    wpli_obs = np.array(wpli_obs)

    return wpli_obs



In [ ]:
# Import LOOK UP TABLE - RBD 
lut_rbd = pd.read_excel('fname_rbd').set_index('ID')

# Set frequency limits
f_min = 2
f_max = 90
scale_freq='log'

#### Import wPLI

In [ ]:
wpliR1 = extract_wpli_new(sorted(glob.glob(os.path.join('wpli_r1'))))
wpliR2 = extract_wpli_new(sorted(glob.glob(os.path.join('wpli_r2'))))

In [ ]:

# Extract information about conveterted and not converted patients
name_lateRBD_r1 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'B') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 1) ].index.to_list()
name_lateRBD_r1 = [i.split('_', 1)[0] for i in name_lateRBD_r1]
name_lateRBD_r2 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'FU1') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 1)  ].index.to_list()
name_lateRBD_r2 = [i.split('_', 1)[0] for i in name_lateRBD_r2]

name_earlyRBD_r1 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'B') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 2) ].index.to_list()
name_earlyRBD_r1 = [i.split('_', 1)[0] for i in name_earlyRBD_r1]
name_earlyRBD_r2 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'FU1') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 2) ].index.to_list()
name_earlyRBD_r2 = [i.split('_', 1)[0] for i in name_earlyRBD_r2]

name_sRBD_r1 = lut_rbd.loc[(lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'B') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 0)].index.to_list()
name_sRBD_r1 = [i.split('_', 1)[0] for i in name_sRBD_r1]
name_sRBD_r2 = lut_rbd.loc[(lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'FU1') & (lut_rbd['REC'] == 'B') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 0)].index.to_list()
name_sRBD_r2 = [i.split('_', 1)[0] for i in name_sRBD_r2]

#### Compute local graph metrics

In [ ]:
nsR1, nCh, _, nF = wpliR1.shape
nsR2, _, _, _ = wpliR2.shape

centralityR1 = np.zeros((nsR1, nCh, nF))
ccR1 = np.zeros_like(centralityR1)
strR1 = np.zeros_like(centralityR1)

centralityR2 = np.zeros((nsR2, nCh, nF))
ccR2 = np.zeros_like(centralityR2)
strR2 = np.zeros_like(centralityR2)

for iS in range(nsR1):
    for iF in range(nF):
        centralityR1[iS, :, iF], ccR1[iS, :, iF], strR1[iS,:,iF] = graph_metrics(wpliR1[iS,:,:,iF])

        
for iS in range(nsR2):
    for iF in range(nF):
        centralityR2[iS, :, iF], ccR2[iS, :, iF], strR2[iS,:,iF] = graph_metrics(wpliR2[iS,:,:,iF])

#### Import fEI

In [ ]:
name_subj_r1, _, fei_r1 = extract_fei_new(sorted(glob.glob(os.path.join('fei_r1'))))
name_subj_r2, _, fei_r2 = extract_fei_new(sorted(glob.glob(os.path.join('fei_r2'))))

In [ ]:
idx_r1_lateRBD = [name_subj_r1.index(x) for x in name_lateRBD_r1 if x in name_subj_r1]
idx_r2_lateRBD = [name_subj_r2.index(x) for x in name_lateRBD_r2 if x in name_subj_r2]

idx_r1_earlyRBD = [name_subj_r1.index(x) for x in name_earlyRBD_r2 if x in name_subj_r1]
idx_r2_earlyRBD = [name_subj_r2.index(x) for x in name_earlyRBD_r2 if x in name_subj_r2]

idx_r1_sRBD = [name_subj_r1.index(x) for x in name_sRBD_r1 if x in name_subj_r1]
idx_r2_sRBD = [name_subj_r2.index(x) for x in name_sRBD_r2 if x in name_subj_r2]

In [ ]:
freq_vals, _ = create_frequency_axis(f_min=f_min, f_max=f_max, scale_freq =scale_freq)

In [ ]:
freq_vals = freq_vals[:30]
fei_r1 = fei_r1[:,:30]
fei_r2 = fei_r2[:,:30]
fei = np.concatenate((fei_r1,fei_r2))
centralityR1 = np.mean(centralityR1, axis=1)
centralityR2 = np.mean(centralityR2, axis=1)
centrality = np.concatenate((centralityR1,centralityR2))
ccR1 = np.nanmean(ccR1[:,:30], axis=1)
ccR2 = np.nanmean(ccR2[:,:30], axis=1)
cc = np.concatenate((ccR1,ccR2))
strR1 = np.nanmean(strR1[:,:30], axis=1)
strR2 = np.nanmean(strR2[:,:30], axis=1)
strenght = np.concatenate((strR1,strR2))

In [ ]:
def split_freq_range(eeg_measure):
    
    eeg_delta = eeg_measure[:, :7].mean(axis=1)
    eeg_theta = eeg_measure[:, 7:11].mean(axis=1)
    eeg_alpha = eeg_measure[:, 11:16].mean(axis=1)
    eeg_beta = eeg_measure[:, 16:22].mean(axis=1)
    eeg_gamma = eeg_measure[:, 22:30].mean(axis=1)

    return eeg_delta, eeg_theta, eeg_alpha, eeg_beta, eeg_gamma

In [ ]:
cent_delta, cent_theta, cent_alpha, cent_beta, cent_gamma = split_freq_range(centrality)
cc_delta, cc_theta, cc_alpha, cc_beta, cc_gamma = split_freq_range(cc)
str_delta, str_theta, str_alpha, str_beta, str_gamma = split_freq_range(strenght)
fei_delta, fei_theta, fei_alpha, fei_beta, fei_gamma = split_freq_range(fei)

In [ ]:
df = pd.read_csv('fname_rbd').set_index('Unnamed: 0')
df.loc[df['Sex']==0, 'Sex'] = 'M'
df.loc[df['Sex']==1, 'Sex'] = 'F'

In [ ]:
from sklearn.preprocessing import StandardScaler

ddf = pd.DataFrame()

ddf['centdelta'] = cent_delta
ddf['centtheta'] = cent_theta
ddf['centalpha'] = cent_alpha
ddf['centbeta'] = cent_beta
ddf['centgamma'] = cent_gamma

ddf['ccdelta'] = cc_delta
ddf['cctheta'] = cc_theta
ddf['ccalpha'] = cc_alpha
ddf['ccbeta'] = cc_beta
ddf['ccgamma'] = cc_gamma

ddf['strdelta'] = str_delta
ddf['strtheta'] = str_theta
ddf['stralpha'] = str_alpha
ddf['strbeta'] = str_beta
ddf['strgamma'] = str_gamma

ddf['fEIdelta'] = fei_delta
ddf['fEItheta'] = fei_theta
ddf['fEIalpha'] = fei_alpha
ddf['fEIbeta'] = fei_beta
ddf['fEIgamma'] = fei_gamma

scaler = StandardScaler()
df_standardized = pd.DataFrame(scaler.fit_transform(ddf.iloc[:,:21]), columns=ddf.columns[:21])

df_standardized['Sex'] = df['Sex'].tolist()
df_standardized['Age'] = df['Age'].tolist()
df_standardized['Subjects'] = df['Subjects'].tolist()


In [ ]:
from sklearn.preprocessing import StandardScaler
from statsmodels.formula.api import mixedlm

freqs = ['delta', 'theta', 'alpha', 'beta', 'gamma']

for freq in freqs:

    formula = f"fEI{freq} ~ cent{freq} + str{freq} + cc{freq} + Age + Sex"

    model = mixedlm(formula, df_standardized, groups=df_standardized["Subjects"]).fit()

    print(model.summary().tables[1])

In [ ]:

def scatter_plot(axs, df_base, df_fup, x_df, y_df, idx_r1_sRBD, idx_r1_lateRBD, idx_r1_earlyRBD, idx_r2_sRBD, idx_r2_lateRBD, idx_r2_earlyRBD, fei_r1, fei_r2, wpli_r1, wpli_r2, cmap):
    
    m_size=8
    a_size=8
    l_size=10

    sns.regplot(x=x_df, y=y_df, data=df_base, fit_reg=True, ci=95, n_boot=1000, 
            scatter_kws={'color':'white', 's':m_size}, line_kws={'color': 'darkgray'}, ax=axs)
    
    sns.regplot(x=x_df, y=y_df, data=df_fup, fit_reg=True, ci=95, n_boot=1000, 
            scatter_kws={'color':'white', 's':m_size}, line_kws={'color': 'dimgray'}, ax=axs)


    # sRBD
    axs.scatter(wpli_r1[idx_r1_sRBD], fei_r1[idx_r1_sRBD], marker='o', color=cmap[2], 
                label='nc-iRBD', s=m_size)
    axs.scatter(wpli_r2[idx_r2_sRBD], fei_r2[idx_r2_sRBD], marker='o', color=cmap[2], s=m_size)

    # late RBD 
    axs.scatter(wpli_r1[idx_r1_lateRBD], fei_r1[idx_r1_lateRBD], marker='^', 
                color=cmap[1], label='late-iRBD', s=m_size) 
    axs.scatter(wpli_r2[idx_r2_lateRBD], 
                fei_r2[idx_r2_lateRBD], marker='^', 
                color=cmap[1], s=m_size) 

    # early RBD
    axs.scatter(wpli_r1[idx_r1_earlyRBD], 
                fei_r1[idx_r1_earlyRBD], marker='X', color='red', 
                label='early-iRBD', s=m_size)
    axs.scatter(wpli_r2[idx_r2_earlyRBD], 
                fei_r2[idx_r2_earlyRBD], marker='X', color='red', 
                s=m_size) 

    axs.tick_params(labelsize=a_size)
    axs.set_ylabel(y_df, fontsize=l_size)
    axs.set_xlabel(x_df, fontsize=l_size)


    axs.spines['top'].set_visible(False)
    axs.spines['right'].set_visible(False)

In [ ]:

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2,2, figsize=(17/2.54, 8.6/2.54), layout='constrained')

df_base = pd.DataFrame({'Strength (15-30) Hz':str_beta[:nsR1],'Clustering Coefficient (15-30) Hz':cc_beta[:nsR1], 'fEI (15-30) Hz':fei_beta[:nsR1], 
                        'Strength (30-70) Hz':str_gamma[:nsR1],'Clustering Coefficient (30-70) Hz':cc_gamma[:nsR1], 'fEI (30-70) Hz':fei_gamma[:nsR1]})

df_fup = pd.DataFrame({'Strength (15-30) Hz':str_beta[nsR1:],'Clustering Coefficient (15-30) Hz':cc_beta[nsR1:], 'fEI (15-30) Hz':fei_beta[nsR1:], 
                        'Strength (30-70) Hz':str_gamma[nsR1:],'Clustering Coefficient (30-70) Hz':cc_gamma[nsR1:], 'fEI (30-70) Hz':fei_gamma[nsR1:]})


scatter_plot(ax1, df_base, df_fup, 'Clustering Coefficient (15-30) Hz', 'fEI (15-30) Hz', 
             idx_r1_sRBD, idx_r1_lateRBD, idx_r1_earlyRBD, idx_r2_sRBD, idx_r2_lateRBD, idx_r2_earlyRBD,
             fei_beta[:nsR1], fei_beta[nsR1:], cc_beta[:nsR1], cc_beta[nsR1:], cmap)

scatter_plot(ax2, df_base, df_fup, 'Strength (15-30) Hz', 'fEI (15-30) Hz', 
             idx_r1_sRBD, idx_r1_lateRBD, idx_r1_earlyRBD, idx_r2_sRBD, idx_r2_lateRBD, idx_r2_earlyRBD,
             fei_beta[:nsR1], fei_beta[nsR1:], str_beta[:nsR1], str_beta[nsR1:], cmap)

scatter_plot(ax3, df_base, df_fup, 'Clustering Coefficient (30-70) Hz', 'fEI (30-70) Hz', 
             idx_r1_sRBD, idx_r1_lateRBD, idx_r1_earlyRBD, idx_r2_sRBD, idx_r2_lateRBD, idx_r2_earlyRBD,
             fei_gamma[:nsR1], fei_gamma[nsR1:], cc_gamma[:nsR1], cc_gamma[nsR1:], cmap)

scatter_plot(ax4, df_base, df_fup, 'Strength (30-70) Hz', 'fEI (30-70) Hz', 
             idx_r1_sRBD, idx_r1_lateRBD, idx_r1_earlyRBD, idx_r2_sRBD, idx_r2_lateRBD, idx_r2_earlyRBD,
             fei_gamma[:nsR1], fei_gamma[nsR1:], str_gamma[:nsR1], str_gamma[nsR1:], cmap)
